In [60]:
"""
Memory-efficient NPI data loading options.
The full file is ~11 GB / 9.4M rows — loading all at once crashes the kernel.
Use one of the approaches below.
"""
import pandas as pd

CSV_PATH = "npidata_pfile_20050523-20260208.csv"

# Columns to load
COLS = [
    "NPI",
    "Provider Organization Name (Legal Business Name)",
    "Provider First Line Business Mailing Address",
    "Provider Business Mailing Address State Name",
    "Provider First Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Mailing Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)",
    "Healthcare Provider Taxonomy Code_1"
]

## All column names in the CSV

In [66]:
# # Print all 330 column names (read header only — no full load)
# import csv
# with open(CSV_PATH, "r", encoding="utf-8") as f:
#     cols = next(csv.reader(f))
# df_cols = pd.DataFrame([(i+1, c) for i, c in enumerate(cols)], columns=["#", "Column Name"])
# pd.set_option("display.max_rows", 400)
# print(f"Total columns: {len(cols)}")
# df_cols

## Option 1: Load only essential columns (recommended)
Reduces memory by ~80–90% by loading a subset of columns.

In [79]:
COUNTRY_COL = "Provider Business Practice Location Address Country Code (If outside U.S.)"
dtype_map = {
    "NPI": "str",
    "Provider Business Practice Location Address Postal Code": "str",
    "Provider Business Mailing Address Postal Code": "str",
}

# Chunked read + filter for US only (avoids loading non-US into memory)
chunks = []
for chunk in pd.read_csv(CSV_PATH, usecols=COLS, dtype=dtype_map, chunksize=500_000):
    chunks.append(chunk[chunk[COUNTRY_COL] == "US"])
df = pd.concat(chunks, ignore_index=True)
# Drop PR (Puerto Rico) and VI (U.S. Virgin Islands)
state_col = "Provider Business Mailing Address State Name"
df = df[~df[state_col].isin(["PR", "VI"])].copy()

# One-hot encoding (BEFORE taxonomy filter) — use Walgreens column for kept/filtered-out focus
org_col = "Provider Organization Name (Legal Business Name)"
RETAILER_COLS = [("Walgreens", "Walgreen"), ("CVS", "CVS"), ("Safeway", "Safeway"), ("Pathmark", "Pathmark"), ("Kaiser_Permanente", "Kaiser Permanente"), ("Kroger", "Kroger"), ("ShopRite", "ShopRite"), ("Costco", "Costco"), ("Health_Mart", "Health Mart"), ("Good_Neighbor", "Good Neighbor"), ("Walmart", "Walmart"), ("Rite_Aid", "Rite Aid")]
for col_name, pattern in RETAILER_COLS:
    df[col_name] = df[org_col].str.contains(pattern, case=False, na=False).astype(int)
df["non_chain"] = (df[[c for c, _ in RETAILER_COLS]].sum(axis=1) == 0).astype(int)

# Keep only providers with Healthcare Provider Taxonomy Code_1 matching (X = literal)
TAX_COL = "Healthcare Provider Taxonomy Code_1"
allowed_tax = ["333600000X", "3336C0003X", "3336I0012X", "3336M0003X", "3336L0003X", "3336C0002X", "332B00000X"]

# Taxonomy counts for df_kept / df_out (all providers)
tax_counts_all = df[TAX_COL].fillna("(blank)").astype(str).value_counts()
tax_counts_kept = tax_counts_all.reindex(allowed_tax, fill_value=0)
tax_counts_filtered_out = tax_counts_all[~tax_counts_all.index.isin(allowed_tax)].sort_values(ascending=False)

df = df[df[TAX_COL].fillna("").astype(str).isin(allowed_tax)].copy()
df = df.reset_index(drop=True)

print(f"Loaded {len(df):,} US rows × {len(df.columns)} columns (excl. PR, VI; pharmacy taxonomy only)")
df.head()

Loaded 68,309 US rows × 24 columns (excl. PR, VI; pharmacy taxonomy only)


,NPI,Provider Organization Name (Legal Business Name),Provider First Line Business Mailing Address,Provider Business Mailing Address State Name,Provider Business Mailing Address Postal Code,Provider First Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Provider Business Practice Location Address Country Code (If outside U.S.),...,Pathmark,Kaiser_Permanente,Kroger,ShopRite,Costco,Health_Mart,Good_Neighbor,Walmart,Rite_Aid,non_chain
0,1033112230,ST JUDE CHILDRENS RESEARCH HOSPITAL INC,262 DANNY THOMAS PLACE MS 0515,TN,381053678,262 DANNY THOMAS PL,MEMPHIS,TN,381053678,US,...,0,0,0,0,0,0,0,0,0,1
1,1811990203,"TURENNE PHARMEDCO, INC.",355 INDUSTRIAL PARK BLVD,AL,361175550,355 INDUSTRIAL PARK BLVD,MONTGOMERY,AL,361175550,US,...,0,0,0,0,0,0,0,0,0,1
2,1326041336,BUTLER PHARMACY INC,11 S ORANGE ST,MO,647301805,11 S ORANGE ST,BUTLER,MO,647301805,US,...,0,0,0,0,0,0,0,0,0,1
3,1861495863,MODERNHEALTH HOLDINGS INC,110 E HUNTINGTON DR,CA,910163415,110 E HUNTINGTON DR,MONROVIA,CA,910163415,US,...,0,0,0,0,0,0,0,0,0,1
4,1447253448,"WILSHIRE PHARMCARE, INC.",26611 CABOT RD. STE B,CA,926537018,4420 E MIRALOMA AVE STE F,ANAHEIM,CA,928071838,US,...,0,0,0,0,0,0,0,0,0,1


In [80]:
org_col = "Provider Organization Name (Legal Business Name)"

# Walgreens or Walgreen (same stem)
retailers = [
    ("Walgreens / Walgreen", "Walgreen"),  # matches both
    ("CVS", "CVS"),
    ("Safeway", "Safeway"),
    ("Pathmark", "Pathmark"),
    ("Kaiser Permanente", "Kaiser Permanente"),
    ("Kroger", "Kroger"),
    ("ShopRite", "ShopRite"),
    ("Costco", "Costco"),
    ("Health Mart", "Health Mart"),
    ("Good Neighbor", "Good Neighbor"),
    ("Walmart", "Walmart"),
    ("Rite Aid", "Rite Aid"),
]

counts = []
for name, pattern in retailers:
    cnt = df[org_col].str.contains(pattern, case=False, na=False).sum()
    counts.append({"Retailer": name, "Count": cnt})

summary = pd.DataFrame(counts)
summary.style.format({"Count": "{:,}"})

,Retailer,Count
0,Walgreens / Walgreen,"2,906"
1,CVS,"4,508"
2,Safeway,88
3,Pathmark,61
4,Kaiser Permanente,6
5,Kroger,289
6,ShopRite,95
7,Costco,629
8,Health Mart,30
9,Good Neighbor,0


In [82]:
# Qty count: KEPT (allowed taxonomy codes)
df_kept = pd.DataFrame({"Taxonomy Code": tax_counts_kept.index, "Qty": tax_counts_kept.values})
print("KEPT:")
display(df_kept)

# Qty count: FILTERED OUT (excluded taxonomy codes)
df_out = pd.DataFrame({"Taxonomy Code": tax_counts_filtered_out.index, "Qty": tax_counts_filtered_out.values})
df_out.to_excel("filtered_out_taxonomy_codes.xlsx", index=False)
print(f"FILTERED OUT ({len(df_out):,} taxonomy codes) — saved to filtered_out_taxonomy_codes.xlsx")
pd.set_option("display.max_rows", None)
display(df_out)
pd.reset_option("display.max_rows")

KEPT (CVS):


,Taxonomy Code,Qty
0,333600000X,76
1,3336C0003X,4430
2,3336I0012X,0
3,3336M0003X,0
4,3336L0003X,1
5,3336C0002X,1


FILTERED OUT (CVS, 45 taxonomy codes) — saved to filtered_out_taxonomy_codes_CVS.xlsx


,Taxonomy Code,Qty
0,332B00000X,3940
1,183500000X,119
2,163WD0400X,62
3,261QM2500X,11
4,261Q00000X,10
5,251S00000X,8
6,251K00000X,7
7,302R00000X,7
8,261QC1500X,7
9,261QM1300X,6


## Create short_ZIP from postal codes (5-digit US ZIP)
US-only data. Practice postal 3–9 digits → 5-digit short_ZIP:
- **3–4 digits:** left-pad to 5 (742→00742, 1742→01742)
- **5 digits:** use as is (10003→10003)
- **6 digits:** first 2, pad to 5
- **7 digits:** first 3, pad to 5 (6591323→00659)
- **8 digits:** first 4, pad to 5 (40110896→04011)
- **9 digits:** first 5 (170331402→17033)

In [63]:
import re

PRACTICE_POSTAL = "Provider Business Practice Location Address Postal Code"
MAILING_POSTAL = "Provider Business Mailing Address Postal Code"

def to_short_zip(val):
    """Convert practice postal to 5-digit short_ZIP per digit-length rules."""
    if pd.isna(val) or val == "":
        return ""
    if isinstance(val, (int, float)):
        val = int(val)
    s = re.sub(r"\D", "", str(val))
    if len(s) == 0:
        return ""
    n = len(s)
    if n <= 4:
        return s.zfill(5)
    if n == 5:
        return s
    if n == 6:
        return s[:2].zfill(5)
    if n == 7:
        return s[:3].zfill(5)
    if n == 8:
        return s[:4].zfill(5)
    return s[:5]  # n >= 9

# Digit-length helper (needed before drop)
def digit_len(val):
    if pd.isna(val) or val == "": return 0
    if isinstance(val, (int, float)): val = int(val)
    return len(re.sub(r"\D", "", str(val)))

digits = df[PRACTICE_POSTAL].apply(digit_len)
df = df[digits > 0].copy().reset_index(drop=True)
digits = df[PRACTICE_POSTAL].apply(digit_len)  # recompute after drop
print(f"Dropped {(digits == 0).sum():,} rows with 0-digit practice postal. Remainder: {len(df):,} rows")

# Force override for specific indices FIRST (before formula)
df["short_ZIP"] = ""
for idx, zip_val in [(6507233, "10087"), (4535337, "08625")]:
    if idx in df.index:
        df.loc[idx, "short_ZIP"] = zip_val

# Apply to_short_zip only for rows still empty
mask = df["short_ZIP"] == ""
df.loc[mask, "short_ZIP"] = df.loc[mask, PRACTICE_POSTAL].apply(to_short_zip)
mask = df["short_ZIP"] == ""
df.loc[mask, "short_ZIP"] = df.loc[mask, MAILING_POSTAL].apply(to_short_zip)

# Digit-length distribution
print("\nDigit-length distribution (practice postal):")
print(digits.value_counts().sort_index().to_string())

# 1-4 digits: left-padded to 5
mask_pad = (digits >= 1) & (digits <= 4)
n_pad = mask_pad.sum()
print(f"\nSanity: 1-4 digits (left-padded to 5) — {n_pad:,} rows found")
if n_pad > 0:
    display(df.loc[mask_pad, [PRACTICE_POSTAL, MAILING_POSTAL, "short_ZIP"]].head(10))
else:
    print("  (US addresses typically have 5 or 9-digit ZIPs)")

# 7-digit and 8-digit data
mask_7 = digits == 7
mask_8 = digits == 8
print(f"\nSanity: 7-digit (first 3, pad to 5) — {mask_7.sum():,} rows")
display(df.loc[mask_7, [PRACTICE_POSTAL, MAILING_POSTAL, "short_ZIP"]].head(10))
print(f"\nSanity: 8-digit (first 4, pad to 5) — {mask_8.sum():,} rows")
display(df.loc[mask_8, [PRACTICE_POSTAL, MAILING_POSTAL, "short_ZIP"]].head(10))

Dropped 0 rows with 0-digit practice postal. Remainder: 68,309 rows

Digit-length distribution (practice postal):
Provider Business Practice Location Address Postal Code
5     7630
9    60679

Sanity: 1-4 digits (left-padded to 5) — 0 rows found
  (US addresses typically have 5 or 9-digit ZIPs)

Sanity: 7-digit (first 3, pad to 5) — 0 rows


,Provider Business Practice Location Address Postal Code,Provider Business Mailing Address Postal Code,short_ZIP



Sanity: 8-digit (first 4, pad to 5) — 0 rows


,Provider Business Practice Location Address Postal Code,Provider Business Mailing Address Postal Code,short_ZIP


## Count retailers in Provider Organization Name
Case-insensitive counts for Walgreens/Walgreen and major pharmacy/grocery chains.

In [ ]:
org_col = "Provider Organization Name (Legal Business Name)"

# Walgreens or Walgreen (same stem)
retailers = [
    ("Walgreens / Walgreen", "Walgreen"),  # matches both
    ("CVS", "CVS"),
    ("Safeway", "Safeway"),
    ("Pathmark", "Pathmark"),
    ("Kaiser Permanente", "Kaiser Permanente"),
    ("Kroger", "Kroger"),
    ("ShopRite", "ShopRite"),
    ("Costco", "Costco"),
    ("Health Mart", "Health Mart"),
    ("Good Neighbor", "Good Neighbor"),
    ("Walmart", "Walmart"),
    ("Rite Aid", "Rite Aid"),
]

counts = []
for name, pattern in retailers:
    cnt = df[org_col].str.contains(pattern, case=False, na=False).sum()
    counts.append({"Retailer": name, "Count": cnt})

summary = pd.DataFrame(counts)
summary.style.format({"Count": "{:,}"})

,Retailer,Count
0,Walgreens / Walgreen,"11,745"
1,CVS,"8,742"
2,Safeway,806
3,Pathmark,104
4,Kaiser Permanente,224
5,Kroger,"1,389"
6,ShopRite,99
7,Costco,"1,885"
8,Health Mart,46
9,Good Neighbor,50


## One-hot encoding for retailers
Columns: 1 if retailer found in org name, 0 otherwise. `non_chain` = 1 when none are found.

In [ ]:
# (column_name, pattern) for one-hot encoding
RETAILER_COLS = [
    ("Walgreens", "Walgreen"),
    ("CVS", "CVS"),
    ("Safeway", "Safeway"),
    ("Pathmark", "Pathmark"),
    ("Kaiser_Permanente", "Kaiser Permanente"),
    ("Kroger", "Kroger"),
    ("ShopRite", "ShopRite"),
    ("Costco", "Costco"),
    ("Health_Mart", "Health Mart"),
    ("Good_Neighbor", "Good Neighbor"),
    ("Walmart", "Walmart"),
    ("Rite_Aid", "Rite Aid"),
]

# Append one-hot columns to original df (in-place)
for col_name, pattern in RETAILER_COLS:
    df[col_name] = df[org_col].str.contains(pattern, case=False, na=False).astype(int)

df["non_chain"] = (df[[c for c, _ in RETAILER_COLS]].sum(axis=1) == 0).astype(int)

# Filtered: Costco only (Costco column == 1)
df_costco = df[df["Costco"] == 1]
print(f"Costco rows: {len(df_costco):,}")
df_costco.head(15)

Costco rows: 1,885


,NPI,Provider Organization Name (Legal Business Name),Provider First Line Business Mailing Address,Provider Business Mailing Address State Name,Provider Business Mailing Address Postal Code,Provider First Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Provider Business Practice Location Address Country Code (If outside U.S.),...,Pathmark,Kaiser_Permanente,Kroger,ShopRite,Costco,Health_Mart,Good_Neighbor,Walmart,Rite_Aid,non_chain
965030,1619087210,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,2207 W COMMONWEALTH AVE,ALHAMBRA,CA,918031302,US,...,0,0,0,0,1,0,0,0,0,0
965031,1497865000,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,17900 NEWHOPE ST,FOUNTAIN VALLEY,CA,92708,US,...,0,0,0,0,1,0,0,0,0,0
965032,1912017526,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,7900 W QUINCY AVE,LITTLETON,CO,80123,US,...,0,0,0,0,1,0,0,0,0,0
965033,1558471169,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,200 FEDERAL RD,BROOKFIELD,CT,06804,US,...,0,0,0,0,1,0,0,0,0,0
965035,1528178134,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,4901 GATE PKWY,JACKSONVILLE,FL,32246,US,...,0,0,0,0,1,0,0,0,0,0
965036,1982714598,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,645 BARRETT PKWY,KENNESAW,GA,30144,US,...,0,0,0,0,1,0,0,0,0,0
965038,1316057920,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,505 W ARMY TRAIL RD,BLOOMINGDALE,IL,60108,US,...,0,0,0,0,1,0,0,0,0,0
965039,1043320658,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,250 N RANDALL RD,LAKE IN THE HILLS,IL,60156,US,...,0,0,0,0,1,0,0,0,0,0
965040,1942310560,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,71 2ND AVE,WALTHAM,MA,02451,US,...,0,0,0,0,1,0,0,0,0,0
965041,1477663094,COSTCO WHOLESALE CORPORATION,PO BOX 34300,WA,981241300,12011 TECHNOLOGY DR,EDEN PRAIRIE,MN,55344,US,...,0,0,0,0,1,0,0,0,0,0
